# T3 — CI/CD com GitHub Actions

**Objetivo:** automatizar o caminho do `git push` até a imagem publicada. Nenhum passo manual, e nada chega
ao registro de imagens sem passar pelos testes.

```
git push ─► testes (pytest) ─► build da imagem ─► teste do container (smoke test) ─► push para o GitHub Container Registry
            └──────── CI ────────┘ └────────────────── CD ──────────────────┘
```

| Conceito | O que é |
|---|---|
| **Workflow** | o arquivo `.github/workflows/*.yml`: o que roda e quando |
| **Job** | um conjunto de passos em uma máquina nova (*runner*) |
| **Step** | um comando (`run`) ou uma ação pronta (`uses`) |
| **Secret** | um valor guardado pelo GitHub, entregue ao job sem aparecer no log |
| **GitHub Container Registry** | o registro de imagens do GitHub: `ghcr.io/<usuário>/<repositório>` |

In [1]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(".env", override=True)   # endereço e credenciais do MLflow (T0)

os.makedirs(".github/workflows", exist_ok=True)

---
# Parte 1 · Juntos

## 1. Os secrets

O runner do GitHub não tem o seu `.env`. As 3 variáveis viram 3 *secrets* do repositório, com os mesmos nomes.

🖱️ Antes, na aba **Actions** do seu repositório: se aparecer o aviso de que os workflows estão desativados
(acontece em repositórios criados com *fork*), clique em **I understand my workflows, go ahead and enable them**.

🖱️ No GitHub, no seu repositório: **Settings** → **Secrets and variables** → **Actions** →
**New repository secret**. Crie um para cada linha do seu `.env`:

| Name | Secret |
|---|---|
| `MLFLOW_TRACKING_URI` | `https://dagshub.com/<usuário>/<repositório>.mlflow` |
| `MLFLOW_TRACKING_USERNAME` | o seu usuário do DagsHub |
| `MLFLOW_TRACKING_PASSWORD` | o seu token do DagsHub |

## 2. O primeiro workflow: rodar os testes

Um job só, `testes`: baixa o código, instala as dependências e roda o `pytest` da T1.

In [2]:
%%writefile .github/workflows/ci.yml
name: CI/CD

on:
  push:                   # roda a cada push, em qualquer branch (hoje, a aula_02)
  pull_request:           # e a cada pull request

jobs:
  testes:
    runs-on: ubuntu-latest            # uma máquina nova, criada só para este job
    steps:
      - uses: actions/checkout@v6     # baixa o repositório
      - uses: actions/setup-python@v6
        with:
          python-version: "3.12"
          cache: pip
      - name: Dependências
        run: pip install -r requirements-api.txt -r requirements-dev.txt
      - name: Testes
        run: pytest -q

Writing .github/workflows/ci.yml


## 3. Publicar e acompanhar

In [3]:
%%bash
git add .github/workflows/ci.yml
git commit -m "CI: testes da API"
git push

[aula_02 562ec1c] CI: testes da API
 1 file changed, 19 insertions(+)
 create mode 100644 .github/workflows/ci.yml


To https://github.com/anasegatelli/mlops-facens
   555637a..562ec1c  aula_02 -> aula_02


🖱️ No GitHub, aba **Actions**: o workflow **CI/CD** aparece rodando. Clique nele e abra o job `testes` para ver
cada passo. Em cerca de um minuto ele fica verde ✅.

> 💬 Se o `git push` recusar o arquivo do workflow, veja "Problemas comuns" no `aula02/README.md`.

## 4. O job `imagem`, linha a linha

O próximo job faz no runner o que você fez à mão na T1 e na T2. O professor percorre o arquivo com a turma; a tabela
fica aqui para você consultar na Parte 2.

| À mão (você, no Codespace) | No job `imagem` (no runner do GitHub) | Por quê |
|---|---|---|
| na T1, rodava o `pytest` e só seguia se passasse | `needs: testes` | o job só começa se o job `testes` ficar verde |
| `docker build -t bike-api .` | `docker build -t "$IMAGE:latest" -t "$IMAGE:${GITHUB_SHA::7}" .` | duas tags: `latest` (a mais nova) e os 7 primeiros caracteres do commit |
| `docker run ... --env-file .env bike-api` | `docker run ... -e MLFLOW_TRACKING_URI ... "$IMAGE:latest"` | o runner não tem `.env`: as 3 variáveis vêm dos secrets |
| `curl` no `/health` e no `/predict`, e você lia a resposta | os mesmos `curl`, com `--fail` | se a API não responder, o passo falha e o job fica vermelho: é o **teste do container (smoke test)** |
| (não tinha) | `docker login ghcr.io` e `docker push --all-tags "$IMAGE"` | publica as duas tags no GitHub Container Registry |

O que mais aparece no arquivo:

- **`if: github.event_name == 'push'`**: o job roda a cada push, na `aula_02` ou na `main`. Em *pull request*,
  só os testes rodam.
- **`permissions: packages: write`**: autoriza o job a publicar imagens no GitHub Container Registry.
- **`$IMAGE`**: o passo "Nome da imagem" monta `ghcr.io/<usuário>/<repositório>` em minúsculas e grava em
  `$GITHUB_ENV`, o que deixa a variável valendo nos passos seguintes.
- **`${{ secrets.X }}` e `$X`**: o primeiro o GitHub preenche antes de o passo rodar, e o valor fica escondido
  no log; o segundo é uma variável comum do terminal.
- **`if: always()`** no passo "Logs do container": mostra os logs mesmo quando o teste do container falha. É ali que se
  procura o erro.
- **`secrets.GITHUB_TOKEN`**: um token que o GitHub cria sozinho a cada execução; não precisa cadastrar.

---
# Parte 2 · Sozinhos

## 1. Complete o job `imagem` e publique

A célula abaixo é o `ci.yml` completo: o job `testes` da Parte 1 e o job `imagem` do passo 4. Faltam
**2 trechos**, os dois no passo **Teste do container (smoke test)**. As dicas estão nos comentários `TODO`.

In [4]:
%%writefile .github/workflows/ci.yml
name: CI/CD

on:
  push:                   # roda a cada push, em qualquer branch (hoje, a aula_02)
  pull_request:           # e a cada pull request

jobs:
  testes:
    runs-on: ubuntu-latest            # uma máquina nova, criada só para este job
    steps:
      - uses: actions/checkout@v6     # baixa o repositório
      - uses: actions/setup-python@v6
        with:
          python-version: "3.12"
          cache: pip
      - name: Dependências
        run: pip install -r requirements-api.txt -r requirements-dev.txt
      - name: Testes
        run: pytest -q

  imagem:
    needs: testes                               # só roda se os testes passarem
    if: github.event_name == 'push'             # em pull request, só os testes
    runs-on: ubuntu-latest
    permissions:
      contents: read
      packages: write                           # permite publicar no GitHub Container Registry
    steps:
      - uses: actions/checkout@v6

      - name: Nome da imagem
        # ghcr.io/<usuário>/<repositório>, em minúsculas (o GitHub Container Registry não aceita maiúsculas)
        run: echo "IMAGE=ghcr.io/${GITHUB_REPOSITORY,,}" >> "$GITHUB_ENV"

      - name: Build
        run: docker build -t "$IMAGE:latest" -t "$IMAGE:${GITHUB_SHA::7}" .

      - name: Teste do container (smoke test)
        # O container sobe? Carrega o @champion do DagsHub? Responde uma previsão?
        env:
          MLFLOW_TRACKING_URI: ${{ secrets.MLFLOW_TRACKING_URI }}
          # TODO 1: copie a linha de cima mais duas vezes, trocando URI por USERNAME e depois por PASSWORD
          #   (nos dois lugares da linha: antes dos dois-pontos e dentro do secrets)
          MLFLOW_TRACKING_USERNAME: ${{ secrets.MLFLOW_TRACKING_USERNAME }}
          MLFLOW_TRACKING_PASSWORD: ${{ secrets.MLFLOW_TRACKING_PASSWORD }}
        run: |
          docker run -d --name api -p 8000:8000 \
            -e MLFLOW_TRACKING_URI -e MLFLOW_TRACKING_USERNAME -e MLFLOW_TRACKING_PASSWORD \
            "$IMAGE:latest"
         # curl --fail --silent --retry 12 --retry-delay 5 --retry-all-errors localhost:8000/health
          # TODO 2: o curl do POST /predict da T1 (passo 3 da Parte 1), com --fail --silent no lugar de -s.
          #   --fail faz o curl dar erro se a API responder com erro: é isso que deixa o job vermelho.
          curl --fail --silent -X POST localhost:8000/predict -H "Content-Type: application/json" -d @api/exemplo.json
      - name: Logs do container
        if: always()
        run: docker logs api

      - name: Login no GitHub Container Registry
        run: echo "${{ secrets.GITHUB_TOKEN }}" | docker login ghcr.io -u "${{ github.actor }}" --password-stdin

      - name: Push
        run: docker push --all-tags "$IMAGE"

Overwriting .github/workflows/ci.yml


In [5]:
%%bash
git add .github/workflows/ci.yml
git commit -m "CD: build, teste do container (smoke test) e push da imagem"
git push

[aula_02 d03f007] CD: build, teste do container (smoke test) e push da imagem
 1 file changed, 43 insertions(+)


To https://github.com/anasegatelli/mlops-facens
   562ec1c..d03f007  aula_02 -> aula_02


🖱️ Na aba **Actions**, o job `testes` roda primeiro; o job `imagem` começa quando ele fica verde. Abra o job
`imagem` e o passo **Teste do container (smoke test)**: as respostas do `/health` e do `/predict` aparecem no log.

Quando os dois jobs ficarem verdes, volte à página inicial do repositório: a imagem aparece em **Packages**,
na coluna da direita, com as tags `latest` e os 7 primeiros caracteres do commit.

## 2. Rode a imagem publicada

Pare o container da T2 e rode a imagem que veio do GitHub Container Registry. É a mesma API, agora construída e testada pelo pipeline.

In [6]:
%%bash
 docker stop bike-api && docker rm bike-api
IMAGE=ghcr.io/$(echo "$GITHUB_REPOSITORY" | tr "A-Z" "a-z")
echo "$IMAGE"
 echo "$GITHUB_TOKEN" | docker login ghcr.io -u "$GITHUB_USER" --password-stdin
 docker pull "$IMAGE:latest"
# TODO: o docker run da T2 (passo 5 da Parte 1), trocando o bike-api do final por "$IMAGE:latest".
#   Depois espere 15 segundos (sleep 15) e consulte o /health.
docker run -d --name bike-api -p 8000:8000 \
  -e MLFLOW_TRACKING_URI -e MLFLOW_TRACKING_USERNAME -e MLFLOW_TRACKING_PASSWORD \
  "$IMAGE:latest"
sleep 15
curl -s localhost:8000/health

bike-api
bike-api
ghcr.io/anasegatelli/mlops-facens
Login Succeeded


Error response from daemon: failed to resolve reference "ghcr.io/anasegatelli/mlops-facens:latest": ghcr.io/anasegatelli/mlops-facens:latest: not found
Unable to find image 'ghcr.io/anasegatelli/mlops-facens:latest' locally
docker: Error response from daemon: failed to resolve reference "ghcr.io/anasegatelli/mlops-facens:latest": ghcr.io/anasegatelli/mlops-facens:latest: not found

Run 'docker run --help' for more information


{"status":"ok","model":"bike-demand","model_version":"6"}

## 3. Pull request e merge na `main`

O trabalho da Aula 2 está todo na `aula_02`. Ele entra na `main` por um *pull request*: o GitHub roda os
testes no PR, e só depois vem o merge.

🖱️ No GitHub, no seu repositório:

1. Na faixa **aula_02 had recent pushes**, clique em **Compare & pull request**. Se a faixa não aparecer: aba
   **Pull requests** → **New pull request**, com *base* `main` e *compare* `aula_02`.
2. Confira o topo da página: o **base repository** precisa ser o **seu** repositório. Em um *fork*, o GitHub
   sugere o repositório da disciplina; troque se for o caso.
3. **Create pull request**. Na página do PR, o job `testes` roda; o `imagem` aparece como pulado (*skipped*).
4. Com os testes verdes: **Merge pull request** → **Confirm merge**.
5. Aba **Actions**: o merge é um push na `main`, então os dois jobs rodam de novo e publicam a imagem com o
   hash do commit do merge.

Volte o Codespace para a `main`, que agora tem a Aula 2 inteira. O Desafio 2 começa daqui.

In [7]:
%%bash
git switch main
git pull
git branch --show-current
git log --oneline -3

Switched to branch 'main'


Your branch is up to date with 'origin/main'.


From https://github.com/anasegatelli/mlops-facens
   fe7a53e..b2e46bc  main       -> origin/main


Updating fe7a53e..b2e46bc
Fast-forward
 .devcontainer/codespace.Dockerfile  |   6 +
 .devcontainer/devcontainer.json     |  11 +-
 materiais-complementares/docker.md  | 718 ++++++++++++++++++++++++++++++++++++
 materiais-complementares/fastapi.md | 694 ++++++++++++++++++++++++++++++++++
 4 files changed, 1428 insertions(+), 1 deletion(-)
 create mode 100644 .devcontainer/codespace.Dockerfile
 create mode 100644 materiais-complementares/docker.md
 create mode 100644 materiais-complementares/fastapi.md
main
b2e46bc Merge branch 'PauloCasmal:main' into main
224b9d6 Adicionando materiais complementares da aula 2
9bd098b Devcontainer: Docker no Codespace para a Aula 2


---
## Para ir além: um portão de qualidade para o modelo

Os testes da T1 conferem a API com um modelo falso. Um CI de ML também confere o **modelo**: o job abaixo baixa
os dados de teste do remote do DVC e exige que o `@champion` fique abaixo de um RMSE máximo.

In [ ]:
%%writefile scripts/quality_gate.py
"""Portão de qualidade: o @champion precisa ter RMSE abaixo do limite no teste atual.

    python scripts/quality_gate.py --max-rmse 80
"""
import argparse
import sys

import mlflow
import pandas as pd

sys.path.insert(0, "src")
from bike import MODEL_NAME, regression_metrics, xy  # noqa: E402

parser = argparse.ArgumentParser()
parser.add_argument("--max-rmse", type=float, required=True)
args = parser.parse_args()

X, y = xy(pd.read_csv("data/split/test.csv"))
model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
rmse = regression_metrics(y, model.predict(X))["rmse"]

print(f"RMSE do @champion: {rmse:.1f} (limite: {args.max_rmse})")
sys.exit(0 if rmse <= args.max_rmse else 1)

In [ ]:
%%bash
python scripts/quality_gate.py --max-rmse 80

Para ligar o portão ao pipeline, acrescente este job ao `ci.yml` (dentro de `jobs:`) e inclua `modelo` no
`needs` do job `imagem`: `needs: [testes, modelo]`.

```yaml
  modelo:
    runs-on: ubuntu-latest
    env:
      MLFLOW_TRACKING_URI: ${{ secrets.MLFLOW_TRACKING_URI }}
      MLFLOW_TRACKING_USERNAME: ${{ secrets.MLFLOW_TRACKING_USERNAME }}
      MLFLOW_TRACKING_PASSWORD: ${{ secrets.MLFLOW_TRACKING_PASSWORD }}
    steps:
      - uses: actions/checkout@v6
      - uses: actions/setup-python@v6
        with:
          python-version: "3.12"
          cache: pip
      - run: pip install -r requirements.txt
      - name: Dados de teste (remote do DVC no DagsHub)
        run: |
          dvc remote modify origin --local auth basic
          dvc remote modify origin --local user "$MLFLOW_TRACKING_USERNAME"
          dvc remote modify origin --local password "$MLFLOW_TRACKING_PASSWORD"
          dvc pull data/split
      - name: Métrica mínima do @champion
        run: python scripts/quality_gate.py --max-rmse 80
```